# 07. 행정경계 전처리

2025년 2분기 SGIS 시군구 경계 252개를 프로젝트의 229개 분석단위로 통합합니다.

**입력**
- `../data_raw/국가데이터처_SGIS 행정구역 통계 및 경계_20250630.zip`
- `../data_raw/sgis_sigungu_252_to_229_mapping.csv`

**출력**
- `../data_processed/sigungu_229_2025.gpkg`

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
import geopandas as gpd

data_raw = Path("../data_raw")
zip_path = data_raw / "국가데이터처_SGIS 행정구역 통계 및 경계_20250630.zip"
extract_dir = data_raw / "sgis_2025"
extract_dir.mkdir(parents=True, exist_ok=True)

print("ZIP 존재:", zip_path.exists())

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

shp_files = list(
    extract_dir.rglob("bnd_sigungu_00_2025_2Q.shp")
)

if len(shp_files) != 1:
    raise ValueError(f"시군구 shp 파일을 1개 찾을 것으로 예상했지만 {len(shp_files)}개 발견")

print("경계 파일:", shp_files[0])

## 1) SGIS 경계와 229개 매핑표 결합

In [ ]:
sigungu = gpd.read_file(shp_files[0])

mapping = pd.read_csv(
    data_raw / "sgis_sigungu_252_to_229_mapping.csv"
)

sigungu["SIGUNGU_CD"] = (
    sigungu["SIGUNGU_CD"].astype(str).str.zfill(5)
)
mapping["SIGUNGU_CD"] = (
    mapping["SIGUNGU_CD"].astype(str).str.zfill(5)
)

sigungu_mapped = sigungu.merge(
    mapping[["SIGUNGU_CD", "SIDO_CD", "SIDO_NM", "sgg_nm_229"]],
    on="SIGUNGU_CD",
    how="left",
    validate="one_to_one"
)

print("원본 경계:", len(sigungu))
print("매핑 실패:", sigungu_mapped["sgg_nm_229"].isna().sum())

## 2) 252개 → 229개 경계 통합

In [ ]:
sigungu_229 = sigungu_mapped.dissolve(
    by=["SIDO_NM", "sgg_nm_229"],
    as_index=False
)

sigungu_229_final = (
    sigungu_229[["SIDO_NM", "sgg_nm_229", "geometry"]]
    .rename(columns={"sgg_nm_229": "SIGUNGU_NM"})
    .copy()
)

print("통합 후 지역 수:", len(sigungu_229_final))
print("좌표계:", sigungu_229_final.crs)
print("빈 geometry:", sigungu_229_final.geometry.isna().sum())
print("유효하지 않은 geometry:",
      (~sigungu_229_final.geometry.is_valid).sum())

assert len(sigungu_229_final) == 229

In [ ]:
output_dir = Path("../data_processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "sigungu_229_2025.gpkg"

sigungu_229_final.to_file(
    output_path,
    layer="sigungu_229",
    driver="GPKG"
)

print("저장 완료:", output_path)